In [1]:
import pandas as pd

census = pd.read_csv("US_Census_Annual_Estimates.csv")
google = pd.read_csv("Google_Search_Monthly_Activity.csv")

print("US Census Dataset:")
display(census.head())

print("Google Search Dataset:")
display(google.head())

US Census Dataset:


,YEAR,LocationAbbr,LocationDesc,TopicType,TopicDesc,DataSource,Data_Value_Type,Data_Value,Sex,Age,...,Source_File_USCB,Data_Pulled,LocationID,TopicTypeId,TopicId,MeasureId,StratificationID1,StratificationID2,SubMeasureID,DisplayOrder
0,2019,AL,Alabama,Census,Census - Population,United States Census Bureau,Count,"4,876,250",Total,Total,...,ACS S0101 AGE AND SEX,3/26/2021 0:00,1.0,CEN,100CEN,100POP,1GEN,8AGE,CEN01,1
1,2019,AL,Alabama,Census,Census - Population,United States Census Bureau,Count,"3,779,874",Total,18+,...,ACS S0101 AGE AND SEX,3/26/2021 0:00,1.0,CEN,100CEN,100POP,1GEN,10AGE,CEN02,2
2,2019,AL,Alabama,Census,Census - Population,United States Census Bureau,Count,"1,096,376",Total,<18,...,ACS S0101 AGE AND SEX,3/26/2021 0:00,1.0,CEN,100CEN,100POP,1GEN,11AGE,CEN03,3
3,2019,AL,Alabama,Census,Census - Population,United States Census Bureau,Count,"2,359,355",Male,Total,...,ACS S0101 AGE AND SEX,3/26/2021 0:00,1.0,CEN,100CEN,100POP,2GEN,8AGE,CEN04,4
4,2019,AL,Alabama,Census,Census - Population,United States Census Bureau,Count,"1,799,897",Male,18+,...,ACS S0101 AGE AND SEX,3/26/2021 0:00,1.0,CEN,100CEN,100POP,2GEN,10AGE,CEN05,5


Google Search Dataset:


,Month,Search_Count
0,2025-06,1
1,2025-08,62
2,2025-09,184
3,2025-10,128
4,2025-11,129


In [2]:
print("=== US CENSUS DATA QUALITY ===")
print("Rows and columns:", census.shape)

print("\nMissing values:")
print(census.isnull().sum())

print("\nDuplicate rows:", census.duplicated().sum())

print("\nData types:")
print(census.dtypes)

print("\n=== GOOGLE SEARCH DATA QUALITY ===")
print("Rows and columns:", google.shape)

print("\nMissing values:")
print(google.isnull().sum())

print("\nDuplicate rows:", google.duplicated().sum())

print("\nData types:")
print(google.dtypes)

=== US CENSUS DATA QUALITY ===
Rows and columns: (477, 21)

Missing values:
YEAR                   0
LocationAbbr           0
LocationDesc           0
TopicType              0
TopicDesc              0
DataSource             0
Data_Value_Type        0
Data_Value             0
Sex                    0
Age                    0
GeoLocation            9
Source_File_USCB       0
Data_Pulled            0
LocationID           422
TopicTypeId            0
TopicId                0
MeasureId              0
StratificationID1      0
StratificationID2      0
SubMeasureID           0
DisplayOrder           0
dtype: int64

Duplicate rows: 0

Data types:
YEAR                   int64
LocationAbbr             str
LocationDesc             str
TopicType                str
TopicDesc                str
DataSource               str
Data_Value_Type          str
Data_Value               str
Sex                      str
Age                      str
GeoLocation              str
Source_File_USCB         str
Data_P

In [3]:
census_clean = census.copy()

census_clean["Data_Value"] = pd.to_numeric(
    census_clean["Data_Value"].astype(str).str.replace(",", "", regex=False),
    errors="coerce"
)

print("Original data type:", census["Data_Value"].dtype)
print("Cleaned data type:", census_clean["Data_Value"].dtype)
print("Values that could not be converted:",
      census_clean["Data_Value"].isna().sum())

display(census_clean[["LocationDesc", "Data_Value"]].head())

Original data type: str
Cleaned data type: int64
Values that could not be converted: 0


,LocationDesc,Data_Value
0,Alabama,4876250
1,Alabama,3779874
2,Alabama,1096376
3,Alabama,2359355
4,Alabama,1799897


In [4]:
print("Missing LocationID by location:")
display(
    census_clean.loc[
        census_clean["LocationID"].isna(),
        ["LocationAbbr", "LocationDesc", "LocationID"]
    ].drop_duplicates().head(20)
)

print("Missing GeoLocation by location:")
display(
    census_clean.loc[
        census_clean["GeoLocation"].isna(),
        ["LocationAbbr", "LocationDesc", "GeoLocation"]
    ].drop_duplicates()
)

print("Total missing LocationID:",
      census_clean["LocationID"].isna().sum())

print("Total missing GeoLocation:",
      census_clean["GeoLocation"].isna().sum())

Missing LocationID by location:


,LocationAbbr,LocationDesc,LocationID
55,CT,Connecticut,NaN
63,DE,Delaware,NaN
72,FL,Florida,NaN
81,GA,Georgia,NaN
90,HI,Hawaii,NaN
99,ID,Idaho,NaN
108,IL,Illinois,NaN
117,IN,Indiana,NaN
126,IA,Iowa,NaN
135,KS,Kansas,NaN


Missing GeoLocation by location:


,LocationAbbr,LocationDesc,GeoLocation
396,US,United States (States and DC),NaN


Total missing LocationID: 422
Total missing GeoLocation: 9


In [5]:
census_clean["LocationID_Missing"] = census_clean["LocationID"].isna()
census_clean["GeoLocation_Missing"] = census_clean["GeoLocation"].isna()

print("Missing LocationID flagged:",
      census_clean["LocationID_Missing"].sum())

print("Missing GeoLocation flagged:",
      census_clean["GeoLocation_Missing"].sum())

census_clean.to_csv("US_Census_Cleaned.csv", index=False)

print("Cleaned Census dataset saved successfully!")

Missing LocationID flagged: 422
Missing GeoLocation flagged: 9
Cleaned Census dataset saved successfully!


In [6]:
google_clean = google.copy()

# Convert Month from text to date
google_clean["Month"] = pd.to_datetime(
    google_clean["Month"], format="%Y-%m"
)

# Find missing months
all_months = pd.date_range(
    start=google_clean["Month"].min(),
    end=google_clean["Month"].max(),
    freq="MS"
)

missing_months = all_months.difference(
    google_clean["Month"]
)

print("Missing months:")
print(missing_months.strftime("%Y-%m").tolist())

print("Month data type:", google_clean["Month"].dtype)

Missing months:
['2025-07']
Month data type: datetime64[us]


In [7]:
google_clean = google_clean.set_index("Month").reindex(all_months)
google_clean.index.name = "Month"
google_clean = google_clean.reset_index()

# Flag months with missing search counts
google_clean["Missing_Search_Count"] = google_clean["Search_Count"].isna()

print("Total months:", len(google_clean))
print("Missing search counts:",
      google_clean["Missing_Search_Count"].sum())

display(google_clean.head(5))

Total months: 18
Missing search counts: 1


,Month,Search_Count,Missing_Search_Count
0,2025-06-01,1.0,False
1,2025-07-01,NaN,True
2,2025-08-01,62.0,False
3,2025-09-01,184.0,False
4,2025-10-01,128.0,False


In [8]:
google_clean.to_csv(
    "Google_Search_Cleaned.csv",
    index=False,
    date_format="%Y-%m"
)

print("Google Search cleaned dataset saved successfully!")
print("Total records:", len(google_clean))
print("Missing search counts:",
      google_clean["Missing_Search_Count"].sum())

Google Search cleaned dataset saved successfully!
Total records: 18
Missing search counts: 1
